# TookaBERT, top four blocks

Continue `PartAI/TookaBERT-Base` from the saved last-block weights. Sign in with the new Colab account. That account's Drive is empty of these weights, so the notebook downloads them from the shared folder `1eyfGAVKe-9nVXnJaQQWWVM2_Zb9fLa4u` onto the Colab disk. Results are written to the new account's `MyDrive/digikala-experiments`.

Those weights already contain the trained classifier and encoder block 11. This notebook unfreezes blocks 8–11, trains one epoch on the full training split, and trains a second epoch only when validation macro F1 beats the saved 0.6722.

Run this on a GPU runtime. One epoch is 115,027 steps and about 15 hours on a T4. Two epochs plus the test pass fit in 40 hours. This notebook does not call MLflow.

The first logged line after step 100 includes the estimated hours for the epoch. Trainer checkpoints stay on the Colab disk. The saved model files are copied to the new Drive before scoring, so a disconnect during scoring still leaves the weights in `runs/PartAI--TookaBERT-Base--top4`.


In [ ]:
MODEL_NAME = "PartAI/TookaBERT-Base"
MAX_LENGTH = 128
EPOCHS = 1
BATCH_SIZE = 16
GRAD_ACCUM = 2
BLOCK_LR = 2e-5
HEAD_LR = 1e-4
TOP_BLOCKS = 4
SEED = 2026

DRIVE_DIR = "/content/drive/MyDrive/digikala-experiments"
WEIGHTS_FOLDER_ID = "1eyfGAVKe-9nVXnJaQQWWVM2_Zb9fLa4u"
START_DIR = "/content/tookabert-start"
DRIVE_FILE_ID = "1vCkaGbSrXtlKUsdOrguUtuEYobT5vFxr"
DATA_ZIP = "/content/data/digikala-comments-preprocessed.csv.zip"
LOCAL_CSV = "/content/data/digikala-comments-preprocessed.csv"

LABELS = ["recommended", "not_recommended", "no_idea"]
LABEL_CODE = {label: code for code, label in enumerate(LABELS)}
SPLIT_SEED = 2026
TRAIN_SHARE = 0.70
VALIDATION_SHARE = 0.15
TEST_SHARE = 0.15


## Drive, the saved weights, and the CSV

Mount the new account's Drive. That mount is where results are saved. Download the TookaBERT `best` folder from the shared link onto `/content/tookabert-start`, then download the preprocessed CSV onto the Colab disk. The notebook reads `body` and `recommendation_status` only.


In [ ]:
from pathlib import Path
import json
import zipfile

import numpy as np
import polars as pl
import torch
from google.colab import drive
from sklearn.metrics import accuracy_score, precision_recall_fscore_support
from sklearn.utils.class_weight import compute_class_weight
from torch.utils.data import Dataset
from transformers import (
    AutoModelForSequenceClassification,
    AutoTokenizer,
    DataCollatorWithPadding,
    Trainer,
    TrainerCallback,
    TrainingArguments,
)

import gdown

drive.mount("/content/drive")
Path(DRIVE_DIR).mkdir(parents=True, exist_ok=True)
(Path(DRIVE_DIR) / "runs").mkdir(parents=True, exist_ok=True)
(Path(DRIVE_DIR) / "logs").mkdir(parents=True, exist_ok=True)
(Path(DRIVE_DIR) / "best").mkdir(parents=True, exist_ok=True)


def slug(model_name: str) -> str:
    return model_name.replace("/", "--")


START_DIR = Path(START_DIR)
if not (START_DIR / "model.safetensors").is_file():
    START_DIR.mkdir(parents=True, exist_ok=True)
    gdown.download_folder(
        id=WEIGHTS_FOLDER_ID,
        output=str(START_DIR),
        quiet=False,
        remaining_ok=True,
    )
if not (START_DIR / "model.safetensors").is_file():
    nested = list(START_DIR.glob("*/model.safetensors"))
    if len(nested) == 1:
        START_DIR = nested[0].parent
if not (START_DIR / "config.json").is_file() or not (START_DIR / "model.safetensors").is_file():
    raise FileNotFoundError(
        "Could not download the shared TookaBERT weights "
        f"{WEIGHTS_FOLDER_ID} into {START_DIR}."
    )
previous_f1 = 0.0
metrics_path = START_DIR / "metrics.json"
if metrics_path.is_file():
    previous_f1 = float(json.loads(metrics_path.read_text()).get("f1") or 0.0)
print("starting weights", START_DIR)
print("results drive", DRIVE_DIR)
print("validation macro F1 to beat", round(previous_f1, 4))

zip_path = Path(DATA_ZIP)
zip_path.parent.mkdir(parents=True, exist_ok=True)
if not zip_path.is_file():
    gdown.download(id=DRIVE_FILE_ID, output=str(zip_path), quiet=False)
if not zip_path.is_file():
    raise FileNotFoundError(f"Could not download Drive file {DRIVE_FILE_ID} to {zip_path}.")
with zipfile.ZipFile(zip_path) as archive:
    member = "digikala-comments-preprocessed.csv"
    names = [
        name
        for name in archive.namelist()
        if name.endswith(".csv") and not name.startswith("__MACOSX")
    ]
    if member not in names:
        if len(names) != 1:
            raise FileNotFoundError(f"Expected one CSV in the zip, found {names}")
        member = names[0]
    info = archive.getinfo(member)
    local_csv = Path(LOCAL_CSV)
    local_csv.parent.mkdir(parents=True, exist_ok=True)
    if (not local_csv.is_file()) or local_csv.stat().st_size != info.file_size:
        print("extracting", member, info.file_size, "bytes")
        with archive.open(info) as source, local_csv.open("wb") as target:
            while True:
                chunk = source.read(8 * 1024 * 1024)
                if not chunk:
                    break
                target.write(chunk)
print("gpu:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu")
print("csv:", local_csv, "bytes", local_csv.stat().st_size)


## The same split as the first TookaBERT run

Each class is shuffled with seed 2026 and cut 70/15/15. The printed counts should match the first run: 3,680,838 train, 788,750 validation, 788,754 test.


In [ ]:
reviews = pl.read_csv(
    LOCAL_CSV,
    columns=["body", "recommendation_status"],
    schema_overrides={"body": pl.String, "recommendation_status": pl.String},
)
labeled = reviews.filter(
    pl.col("body").is_not_null()
    & (pl.col("body").str.strip_chars() != "")
    & pl.col("recommendation_status").is_in(LABELS)
).with_columns(
    pl.col("recommendation_status").replace_strict(LABEL_CODE).alias("label_id")
)

parts = []
for label in LABELS:
    group = labeled.filter(pl.col("recommendation_status") == label)
    shuffled = group.sample(fraction=1.0, shuffle=True, seed=SPLIT_SEED)
    count = shuffled.height
    n_train = int(count * TRAIN_SHARE)
    n_validation = int(count * VALIDATION_SHARE)
    n_test = count - n_train - n_validation
    parts.append(
        shuffled.with_columns(
            pl.Series(
                "split",
                (["train"] * n_train)
                + (["validation"] * n_validation)
                + (["test"] * n_test),
            )
        )
    )
labeled = pl.concat(parts)
train_frame = labeled.filter(pl.col("split") == "train")
valid_frame = labeled.filter(pl.col("split") == "validation")
test_frame = labeled.filter(pl.col("split") == "test")
print("train", train_frame.height, "validation", valid_frame.height, "test", test_frame.height)
print(
    labeled.group_by("split", "recommendation_status")
    .len()
    .sort("split", "recommendation_status")
)


## Top four blocks

Blocks 8, 9, 10, and 11 train at `2e-5`. The classifier trains at `1e-4`. The bottom eight blocks stay frozen. The loss is class-weighted cross-entropy, and the printed precision, recall, and F1 are macro averages.


In [ ]:
class ReviewSet(Dataset):
    def __init__(self, frame, tokenizer, max_length: int):
        self.texts = frame.get_column("body").to_list()
        self.labels = frame.get_column("label_id").to_list()
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, index):
        encoded = self.tokenizer(
            self.texts[index] or "",
            truncation=True,
            max_length=self.max_length,
        )
        encoded["labels"] = int(self.labels[index])
        return encoded


def metric_bundle(actual, predicted):
    precision, recall, f1, _ = precision_recall_fscore_support(
        actual, predicted, labels=[0, 1, 2], average="macro", zero_division=0
    )
    per_precision, per_recall, per_f1, per_support = precision_recall_fscore_support(
        actual, predicted, labels=[0, 1, 2], average=None, zero_division=0
    )
    report = {
        "accuracy": float(accuracy_score(actual, predicted)),
        "precision": float(precision),
        "recall": float(recall),
        "f1": float(f1),
    }
    for index, name in enumerate(LABELS):
        report[f"{name}_precision"] = float(per_precision[index])
        report[f"{name}_recall"] = float(per_recall[index])
        report[f"{name}_f1"] = float(per_f1[index])
        report[f"{name}_support"] = int(per_support[index])
    return report


def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predicted = np.argmax(logits, axis=-1)
    scores = metric_bundle(labels, predicted)
    return {key: scores[key] for key in ("accuracy", "precision", "recall", "f1")}


class WeightedTrainer(Trainer):
    def __init__(self, *args, class_weights=None, **kwargs):
        super().__init__(*args, **kwargs)
        self.class_weights = class_weights

    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        weight = None
        if self.class_weights is not None:
            weight = self.class_weights.to(outputs.logits.device)
        loss = torch.nn.functional.cross_entropy(outputs.logits, labels, weight=weight)
        return (loss, outputs) if return_outputs else loss


class EpochClock(TrainerCallback):
    def on_train_begin(self, args, state, control, **kwargs):
        import time
        self.started = time.perf_counter()

    def on_log(self, args, state, control, logs=None, **kwargs):
        import time
        if state.global_step != 100 or not state.max_steps or not getattr(self, "started", None):
            return
        hours = (time.perf_counter() - self.started) * state.max_steps / 100 / 3600
        print(f"step 100 of {state.max_steps}; this epoch is about {hours:.1f} hours")


def unfreeze_top_blocks(model, n_blocks: int) -> None:
    prefix = "bert"
    for name, param in model.named_parameters():
        param.requires_grad = not name.startswith(prefix + ".")
    layers = model.bert.encoder.layer
    if n_blocks > len(layers):
        raise ValueError(f"TookaBERT has {len(layers)} blocks, requested {n_blocks}.")
    for layer in layers[-n_blocks:]:
        for param in layer.parameters():
            param.requires_grad = True
    for name, param in model.named_parameters():
        if "classifier" in name:
            param.requires_grad = True
    first = len(layers) - n_blocks
    print("training blocks", list(range(first, len(layers))), "and the classifier")


def parameter_groups(model, block_lr: float, head_lr: float):
    head, block = [], []
    for name, param in model.named_parameters():
        if not param.requires_grad:
            continue
        if name.startswith("bert."):
            block.append(param)
        else:
            head.append(param)
    return [
        {"params": block, "lr": block_lr},
        {"params": head, "lr": head_lr},
    ]


def training_arguments(output_dir: str) -> TrainingArguments:
    shared = dict(
        output_dir=output_dir,
        learning_rate=BLOCK_LR,
        per_device_train_batch_size=BATCH_SIZE,
        per_device_eval_batch_size=BATCH_SIZE,
        gradient_accumulation_steps=GRAD_ACCUM,
        num_train_epochs=EPOCHS,
        save_strategy="epoch",
        save_total_limit=1,
        fp16=torch.cuda.is_available(),
        logging_steps=100,
        report_to="none",
        seed=SEED,
        dataloader_num_workers=2,
    )
    try:
        return TrainingArguments(eval_strategy="no", **shared)
    except TypeError:
        return TrainingArguments(evaluation_strategy="no", **shared)


def trainer_token_kwargs(tokenizer):
    import inspect
    if "processing_class" in inspect.signature(Trainer.__init__).parameters:
        return {"processing_class": tokenizer}
    return {"tokenizer": tokenizer}


## Train, then score

Trainer checkpoints stay on the Colab disk. After each finished epoch the cell copies the model files to the new account's `MyDrive/digikala-experiments/runs/PartAI--TookaBERT-Base--top4`. Validation is one pass after the epoch. A second epoch runs only when that macro F1 is higher than the saved last-block score. The test split is scored once. The new account's `best/` folder is written only when the new validation macro F1 wins.


In [ ]:
torch.manual_seed(SEED)
tokenizer = AutoTokenizer.from_pretrained(START_DIR)
model = AutoModelForSequenceClassification.from_pretrained(START_DIR)
unfreeze_top_blocks(model, TOP_BLOCKS)
trainable = sum(param.numel() for param in model.parameters() if param.requires_grad)
print("trainable parameters", trainable)

class_weights = torch.tensor(
    compute_class_weight(
        "balanced",
        classes=np.array([0, 1, 2]),
        y=train_frame.get_column("label_id").to_numpy(),
    ),
    dtype=torch.float,
)
train_set = ReviewSet(train_frame, tokenizer, MAX_LENGTH)
valid_set = ReviewSet(valid_frame, tokenizer, MAX_LENGTH)
test_set = ReviewSet(test_frame, tokenizer, MAX_LENGTH)
collator = DataCollatorWithPadding(tokenizer)
token_kwargs = trainer_token_kwargs(tokenizer)

stage = "top4"
local_run = Path("/content/tookabert-top4")
drive_run = Path(DRIVE_DIR) / "runs" / f"{slug(MODEL_NAME)}--{stage}"
local_run.mkdir(parents=True, exist_ok=True)
drive_run.mkdir(parents=True, exist_ok=True)
groups = parameter_groups(model, BLOCK_LR, HEAD_LR)


def publish(metrics: dict) -> None:
    model.save_pretrained(drive_run)
    tokenizer.save_pretrained(drive_run)
    (drive_run / "metrics.json").write_text(json.dumps(metrics, indent=2))
    print("copied weights to", drive_run)


def make_trainer():
    trainer = WeightedTrainer(
        model=model,
        args=training_arguments(str(local_run)),
        train_dataset=train_set,
        data_collator=collator,
        compute_metrics=compute_metrics,
        class_weights=class_weights,
        callbacks=[EpochClock()],
        **token_kwargs,
    )

    def create_optimizer():
        trainer.optimizer = torch.optim.AdamW(groups, weight_decay=0.01)
        return trainer.optimizer

    trainer.create_optimizer = create_optimizer
    return trainer


def score(frame_set):
    holder = Trainer(
        model=model,
        args=training_arguments("/tmp/tookabert-score"),
        data_collator=collator,
        **token_kwargs,
    )
    prediction = holder.predict(frame_set)
    return metric_bundle(prediction.label_ids, prediction.predictions.argmax(axis=-1))


def save_weights(metrics: dict) -> None:
    publish(metrics)


print("epoch 1")
make_trainer().train()
model.save_pretrained(drive_run)
tokenizer.save_pretrained(drive_run)
print("copied weights to the new Drive before scoring", drive_run)

validation_metrics = score(valid_set)
validation_metrics.update({"split": "validation", "stage": stage, "model_name": MODEL_NAME, "epoch": 1})
save_weights(validation_metrics)
print("validation epoch 1", {key: validation_metrics[key] for key in ("accuracy", "precision", "recall", "f1")})

kept = validation_metrics
if validation_metrics["f1"] > previous_f1:
    print("epoch 2")
    make_trainer().train()
    second = score(valid_set)
    second.update({"split": "validation", "stage": stage, "model_name": MODEL_NAME, "epoch": 2})
    print("validation epoch 2", {key: second[key] for key in ("accuracy", "precision", "recall", "f1")})
    if second["f1"] >= validation_metrics["f1"]:
        kept = second
        save_weights(kept)
    else:
        print("epoch 2 did not beat epoch 1; reloading the epoch 1 weights from the new Drive")
        model = AutoModelForSequenceClassification.from_pretrained(drive_run)
else:
    print(
        "validation macro F1",
        round(validation_metrics["f1"], 4),
        "does not beat",
        round(previous_f1, 4),
        "; stopping before epoch 2",
    )

test_metrics = score(test_set)
test_metrics.update({"split": "test", "stage": stage, "model_name": MODEL_NAME})
(drive_run / "test_metrics.json").write_text(json.dumps(test_metrics, indent=2))
print("test", {key: test_metrics[key] for key in ("accuracy", "precision", "recall", "f1")})

log_path = Path(DRIVE_DIR) / "logs" / f"{slug(MODEL_NAME)}--{stage}.jsonl"
log_path.parent.mkdir(parents=True, exist_ok=True)
with log_path.open("a") as handle:
    handle.write(json.dumps(kept) + "\n")
    handle.write(json.dumps(test_metrics) + "\n")

if kept["f1"] > previous_f1:
    best_dir = Path(DRIVE_DIR) / "best" / slug(MODEL_NAME)
    best_dir.mkdir(parents=True, exist_ok=True)
    for item in drive_run.iterdir():
        if item.is_file():
            (best_dir / item.name).write_bytes(item.read_bytes())
    print("best model updated on the new Drive", best_dir, "macro F1", round(kept["f1"], 4))
else:
    print("new Drive best folder left unchanged")
